In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# 1. Doc du lieu trieu chung. File .xls hien tai thuc chat la CSV text.
data_path = Path('data/transformed_dataset.csv')
if not data_path.exists():
    data_path = Path('data/transformed_dataset.xls')

df_transformed = pd.read_csv(data_path)
df_transformed.columns = [c.strip() for c in df_transformed.columns]

# 2. Tach cot trieu chung, bo cot index/nhan khong duoc dua vao feature.
drop_cols = {'', 'Unnamed: 0', 'Disease', 'Disease_code'}
symptom_cols = [col for col in df_transformed.columns if col not in drop_cols]

# 3. Loai dong trung lap de tranh cung mot mau xuat hien o ca train va test.
before_rows = len(df_transformed)
df_model = df_transformed.drop_duplicates(subset=symptom_cols + ['Disease']).reset_index(drop=True)
removed_rows = before_rows - len(df_model)

X = df_model[symptom_cols]
y = df_model['Disease']

# 4. Chia du lieu co stratify neu moi lop co du mau.
min_class_count = y.value_counts().min()
stratify_target = y if min_class_count >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_target,
)

# 5. Giam overfitting: gioi han do sau cay, tang mau toi thieu moi la,
#    va chi lay mot phan feature o moi lan tach nhanh.
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=8,
    min_samples_leaf=3,
    max_features='sqrt',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

print('Mo hinh da duoc huan luyen thanh cong!')
print(f'Du lieu: {len(df_model)} dong, da loai {removed_rows} dong trung lap')
print(f'So trieu chung: {len(symptom_cols)} | So lop benh: {y.nunique()}')

Mo hinh da duoc huan luyen thanh cong!
Du lieu: 304 dong, da loai 0 dong trung lap
So trieu chung: 131 | So lop benh: 41


In [11]:
import warnings

from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

warnings.filterwarnings(
    'ignore',
    message='The number of unique classes is greater than 50% of the number of samples.*',
)

# Danh gia tren ca train va test de nhin ro dau hieu qua khop.
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
overfit_gap = train_accuracy - test_accuracy

print(f'Accuracy train: {train_accuracy * 100:.2f}%')
print(f'Accuracy test : {test_accuracy * 100:.2f}%')
print(f'Chenhlech train-test: {overfit_gap * 100:.2f}%')
print(f'Precision weighted: {precision_score(y_test, y_test_pred, average="weighted", zero_division=0) * 100:.2f}%')
print(f'Recall weighted   : {recall_score(y_test, y_test_pred, average="weighted", zero_division=0) * 100:.2f}%')
print(f'F1 weighted       : {f1_score(y_test, y_test_pred, average="weighted", zero_division=0) * 100:.2f}%')

print('\nBao cao chi tiet theo tung benh:')
print(classification_report(y_test, y_test_pred, zero_division=0))

Accuracy train: 100.00%
Accuracy test : 100.00%
Chenhlech train-test: 0.00%
Precision weighted: 100.00%
Recall weighted   : 100.00%
F1 weighted       : 100.00%

Bao cao chi tiet theo tung benh:
                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00         1
                                   AIDS       1.00      1.00      1.00         1
                                   Acne       1.00      1.00      1.00         1
                    Alcoholic hepatitis       1.00      1.00      1.00         2
                                Allergy       1.00      1.00      1.00         1
                              Arthritis       1.00      1.00      1.00         1
                       Bronchial Asthma       1.00      1.00      1.00         1
                   Cervical spondylosis       1.00      1.00      1.00         1
                            Chicken pox       1.00      1.00      1.00      

In [12]:
from pathlib import Path

import joblib

# 1. Lưu mô hình vào file có tên là 'random_forest_health_model.pkl'
model_filename = 'model/random_forest_health_model.pkl'
Path(model_filename).parent.mkdir(parents=True, exist_ok=True)
joblib.dump(rf_model, model_filename)

print(f"Mô hình đã được lưu tại: {model_filename}")

Mô hình đã được lưu tại: model/random_forest_health_model.pkl
